In [48]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from loader.trade_loader import load_trades_from_inputs

from scripts.sweep_confirmation_filters import (
    RiskModelConfig,
    parse_float_list,
    ensure_output_dirs,
    load_events,
    validate_raw_trades,
    validate_raw_trade_coverage,
    filter_events_for_combo,
    assign_impulse_groups,
    label_lifecycle_for_representatives,
    summarize_combo,
    plot_histogram,
    plot_summary_heatmap,
    normalize_output_precision,
    combo_id_for_thresholds,
)

In [49]:
EVENTS_PARQUET = "../research/110626/avaxusdc_1541/1541-broad.parquet"
RAW_TRADES = "../storage/avaxusdc/parquet/AVAXUSDC-aggTrades-2025-06_to_2026-05.parquet"
OUTPUT_DIR = "../research/110626/avaxusdc_1541"

REACTION_WINDOW_SECONDS = 10

MFE_THRESHOLDS = [
    0,
    0.00025,
    0.0005,
    0.00075,
    0.001,
    0.0015,
    0.002,
]

EFFICIENCY_THRESHOLDS = [
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
]

MAX_HORIZON_SECONDS = 3600
IMPULSE_GROUP_GAP_SECONDS = 900
PRIMARY_RISK_MODEL = "medium"

TIGHT_STOP_BUFFER_PCT = 0.0005
MEDIUM_STOP_BUFFER_PCT = 0.0010
WIDE_STOP_BUFFER_PCT = 0.0020

WRITE_COMBO_LIFECYCLE_ROWS = True
MAX_HISTOGRAM_R = 10.0

In [50]:
paths = ensure_output_dirs(OUTPUT_DIR, WRITE_COMBO_LIFECYCLE_ROWS)

events_df = load_events(EVENTS_PARQUET, REACTION_WINDOW_SECONDS)
print(f"Loaded events: {len(events_df):,}")

trades_df = validate_raw_trades(load_trades_from_inputs(RAW_TRADES))
print(f"Loaded trades: {len(trades_df):,}")

validate_raw_trade_coverage(
    trades_df=trades_df,
    events_df=events_df,
    max_horizon_seconds=MAX_HORIZON_SECONDS,
)

trades_timestamps = trades_df["timestamp"].to_numpy(dtype=np.int64)
trades_prices = trades_df["price"].to_numpy(dtype=np.float64)

risk_models = [
    RiskModelConfig("tight", TIGHT_STOP_BUFFER_PCT),
    RiskModelConfig("medium", MEDIUM_STOP_BUFFER_PCT),
    RiskModelConfig("wide", WIDE_STOP_BUFFER_PCT),
]

Loaded events: 38,300
Loaded trades: 21,080,601


In [51]:
summary_rows = []
combo_lifecycle = {}

total_combos = len(MFE_THRESHOLDS) * len(EFFICIENCY_THRESHOLDS)
combo_number = 0

for mfe_threshold in MFE_THRESHOLDS:
    for efficiency_threshold in EFFICIENCY_THRESHOLDS:
        combo_number += 1
        combo_id = combo_id_for_thresholds(mfe_threshold, efficiency_threshold)

        print(
            f"[{combo_number}/{total_combos}] {combo_id} "
            f"(mfe >= {mfe_threshold}, eff >= {efficiency_threshold})"
        )

        filtered_df = filter_events_for_combo(
            events_df=events_df,
            reaction_window_seconds=REACTION_WINDOW_SECONDS,
            mfe_threshold=mfe_threshold,
            efficiency_threshold=efficiency_threshold,
        )

        candidate_count_after_filter = len(filtered_df)

        if filtered_df.empty:
            summary_rows.append(
                summarize_combo(
                    combo_id=combo_id,
                    lifecycle_df=pd.DataFrame(),
                    reaction_window_seconds=REACTION_WINDOW_SECONDS,
                    mfe_threshold=mfe_threshold,
                    efficiency_threshold=efficiency_threshold,
                    candidate_count_after_filter=0,
                    representative_count=0,
                    primary_risk_model=PRIMARY_RISK_MODEL,
                )
            )
            continue

        grouped_df = assign_impulse_groups(
            filtered_df,
            impulse_group_gap_seconds=IMPULSE_GROUP_GAP_SECONDS,
        )

        representatives_df = grouped_df.groupby(
            "impulse_group_id",
            sort=False,
            as_index=False,
        ).first()

        lifecycle_df = label_lifecycle_for_representatives(
            representatives_df=representatives_df,
            trades_timestamps=trades_timestamps,
            trades_prices=trades_prices,
            risk_models=risk_models,
            max_horizon_seconds=MAX_HORIZON_SECONDS,
            primary_risk_model=PRIMARY_RISK_MODEL,
            reaction_window_seconds=REACTION_WINDOW_SECONDS,
        )

        lifecycle_df = normalize_output_precision(lifecycle_df)

        summary_row = summarize_combo(
            combo_id=combo_id,
            lifecycle_df=lifecycle_df,
            reaction_window_seconds=REACTION_WINDOW_SECONDS,
            mfe_threshold=mfe_threshold,
            efficiency_threshold=efficiency_threshold,
            candidate_count_after_filter=candidate_count_after_filter,
            representative_count=len(representatives_df),
            primary_risk_model=PRIMARY_RISK_MODEL,
        )

        summary_rows.append(summary_row)
        combo_lifecycle[combo_id] = lifecycle_df

[1/42] mfe_0__eff_0p4 (mfe >= 0, eff >= 0.4)
[2/42] mfe_0__eff_0p5 (mfe >= 0, eff >= 0.5)
[3/42] mfe_0__eff_0p6 (mfe >= 0, eff >= 0.6)
[4/42] mfe_0__eff_0p7 (mfe >= 0, eff >= 0.7)
[5/42] mfe_0__eff_0p8 (mfe >= 0, eff >= 0.8)
[6/42] mfe_0__eff_0p9 (mfe >= 0, eff >= 0.9)
[7/42] mfe_0p00025__eff_0p4 (mfe >= 0.00025, eff >= 0.4)
[8/42] mfe_0p00025__eff_0p5 (mfe >= 0.00025, eff >= 0.5)
[9/42] mfe_0p00025__eff_0p6 (mfe >= 0.00025, eff >= 0.6)
[10/42] mfe_0p00025__eff_0p7 (mfe >= 0.00025, eff >= 0.7)
[11/42] mfe_0p00025__eff_0p8 (mfe >= 0.00025, eff >= 0.8)
[12/42] mfe_0p00025__eff_0p9 (mfe >= 0.00025, eff >= 0.9)
[13/42] mfe_0p0005__eff_0p4 (mfe >= 0.0005, eff >= 0.4)
[14/42] mfe_0p0005__eff_0p5 (mfe >= 0.0005, eff >= 0.5)
[15/42] mfe_0p0005__eff_0p6 (mfe >= 0.0005, eff >= 0.6)
[16/42] mfe_0p0005__eff_0p7 (mfe >= 0.0005, eff >= 0.7)
[17/42] mfe_0p0005__eff_0p8 (mfe >= 0.0005, eff >= 0.8)
[18/42] mfe_0p0005__eff_0p9 (mfe >= 0.0005, eff >= 0.9)
[19/42] mfe_0p00075__eff_0p4 (mfe >= 0.00075, eff

In [52]:
summary_parquet_path = os.path.join(OUTPUT_DIR, "confirmation_sweep_summary.parquet")
summary_csv_path = os.path.join(OUTPUT_DIR, "confirmation_sweep_summary.csv")

summary_df.to_parquet(summary_parquet_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)

print(summary_parquet_path)
print(summary_csv_path)

../research/110626/avaxusdc_1541\confirmation_sweep_summary.parquet
../research/110626/avaxusdc_1541\confirmation_sweep_summary.csv


In [47]:
import pandas as pd
from pathlib import Path

summary_path = Path(OUTPUT_DIR) / "confirmation_sweep_summary.csv"

print(summary_path.resolve())
summary_df = pd.read_csv(summary_path)

summary_df.head(10)[[
    "combo_id",
    "candidate_count_after_filter",
    "representative_count",
    "sample_count",
    "profitable_2R_before_SL_rate",
    "median_medium_max_R_before_stop",
]]

events_df = pd.read_parquet(EVENTS_PARQUET)

print(EVENTS_PARQUET)
print(events_df.shape)
print(events_df["symbol"].value_counts(dropna=False).head())
print(events_df["session_id"].min(), events_df["session_id"].max())

C:\Users\adjop\OneDrive\Documents\golden-goose-project\research\110626\avaxusdc_1522\1522_efficiency\confirmation_sweep_summary.csv
../research/110626/avaxusdc_1522/1522-broad.parquet
(39170, 131)
symbol
AVAXUSDC    39170
Name: count, dtype: int64
2025-06-02 2026-05-31


In [42]:
summary_df = pd.read_csv(f"{OUTPUT_DIR}/confirmation_sweep_summary.csv")

summary_df["reaction_window_seconds"].value_counts()

reaction_window_seconds
30    42
Name: count, dtype: int64

In [15]:
summary_df = pd.DataFrame(summary_rows)

primary_median_col = f"median_{PRIMARY_RISK_MODEL}_max_R_before_stop"

summary_df = summary_df.sort_values(
    [primary_median_col, "profitable_2R_before_SL_rate"],
    ascending=[False, False],
    na_position="last",
    kind="mergesort",
).reset_index(drop=True)

summary_df = normalize_output_precision(summary_df)

summary_df.head(20)

,combo_id,reaction_window_seconds,mfe_threshold_pct,efficiency_threshold,candidate_count_after_filter,representative_count,sample_count,profitable_2R_before_SL_count,profitable_2R_before_SL_rate,stop_before_2R_count,...,max_medium_max_R_before_stop,mean_medium_max_R_within_horizon,median_medium_max_R_within_horizon,p90_medium_max_R_within_horizon,p95_medium_max_R_within_horizon,max_medium_max_R_within_horizon,mean_time_to_2R_seconds,median_time_to_2R_seconds,mean_time_to_stop_seconds,median_time_to_stop_seconds
0,mfe_0p0005__eff_0p7,30,0.00050,0.7,14295,451,451,114,0.252772,242,...,14.594429,1.811548,1.254416,3.593544,5.181360,27.720691,1604.747386,1444.7210,1128.665748,818.3800
1,mfe_0p0005__eff_0p5,30,0.00050,0.5,16800,456,456,116,0.254386,246,...,14.594429,1.883685,1.304798,4.006524,5.550128,27.720691,1546.624833,1348.5120,1107.783549,800.9800
2,mfe_0__eff_0p6,30,0.00000,0.6,30901,164,164,37,0.225610,96,...,9.793991,2.186646,1.395515,4.723361,7.510441,27.720691,1268.197561,878.5110,875.456189,547.1520
3,mfe_0p0005__eff_0p4,30,0.00050,0.4,17793,459,459,115,0.250545,250,...,14.594429,1.941552,1.304871,4.026532,5.682483,27.720691,1544.765562,1348.5120,1093.846659,763.9580
4,mfe_0p0005__eff_0p6,30,0.00050,0.6,15658,454,454,114,0.251101,245,...,14.594429,1.833306,1.296403,3.775168,5.278511,27.720691,1584.670014,1413.4790,1119.234973,812.4740
5,mfe_0p0005__eff_0p9,30,0.00050,0.9,10464,452,452,112,0.247788,243,...,11.577363,1.704291,1.239364,3.578893,4.943665,13.169701,1662.793453,1560.8560,1120.144027,812.4740
6,mfe_0__eff_0p5,30,0.00000,0.5,34136,169,169,40,0.236686,100,...,9.793991,2.423893,1.485468,5.528116,8.747228,27.720691,1178.785935,826.9845,847.755197,536.6530
7,mfe_0__eff_0p4,30,0.00000,0.4,37502,169,169,39,0.230769,103,...,9.793991,2.589566,1.507692,5.614038,9.193757,27.720691,1236.970877,948.9660,810.928067,468.9370
8,mfe_0p0005__eff_0p8,30,0.00050,0.8,12661,454,454,113,0.248899,246,...,11.577363,1.730570,1.239364,3.593292,5.166165,13.169701,1634.093288,1548.1510,1129.036054,821.7330
9,mfe_0__eff_0p7,30,0.00000,0.7,27718,160,160,37,0.231250,93,...,9.793991,2.180701,1.377111,4.721176,7.593135,27.720691,1307.064679,918.2915,882.477972,547.1520


In [34]:
summary_parquet_path = os.path.join(OUTPUT_DIR, "confirmation_sweep_summary.parquet")
summary_csv_path = os.path.join(OUTPUT_DIR, "confirmation_sweep_summary.csv")

summary_df.to_parquet(summary_parquet_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)

print(summary_parquet_path)
print(summary_csv_path)

../research/110626/avaxusdc_1439/1439_efficiency\confirmation_sweep_summary.parquet
../research/110626/avaxusdc_1439/1439_efficiency\confirmation_sweep_summary.csv


In [17]:
combo_dir = os.path.join(OUTPUT_DIR, "combo_lifecycle_rows")
os.makedirs(combo_dir, exist_ok=True)

for combo_id, lifecycle_df in combo_lifecycle.items():
    if not lifecycle_df.empty:
        lifecycle_df.to_parquet(
            os.path.join(combo_dir, f"{combo_id}.parquet"),
            index=False,
        )

In [18]:
def show_summary_heatmap(summary_df, value_column, title):
    pivot = summary_df.pivot_table(
        index="mfe_threshold_pct",
        columns="efficiency_threshold",
        values=value_column,
        aggfunc="first",
    ).sort_index(ascending=True)

    matrix = pivot.to_numpy(dtype=np.float64)

    fig, ax = plt.subplots(figsize=(10, 6))
    image = ax.imshow(matrix, aspect="auto", origin="lower", interpolation="nearest")

    ax.set_title(title)
    ax.set_xlabel("Efficiency threshold")
    ax.set_ylabel("MFE threshold pct")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(x) for x in pivot.columns], rotation=45, ha="right")

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(x) for x in pivot.index])

    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    plt.show()

In [26]:
from pathlib import Path

hist_dir = Path(OUTPUT_DIR) / "histograms"

print("OUTPUT_DIR:", OUTPUT_DIR)
print("hist_dir:", hist_dir)
print("hist_dir absolute:", hist_dir.resolve())
print("hist_dir exists:", hist_dir.exists())

if hist_dir.exists():
    all_files = sorted(hist_dir.glob("*"))
    print("file count:", len(all_files))
    for p in all_files[:50]:
        print(p.name)

OUTPUT_DIR: ../research/110626/1226/1226_efficiency
hist_dir: ..\research\110626\1226\1226_efficiency\histograms
hist_dir absolute: C:\Users\adjop\OneDrive\Documents\golden-goose-project\research\110626\1226\1226_efficiency\histograms
hist_dir exists: True
file count: 0


In [27]:
from pathlib import Path

hist_dir = Path(OUTPUT_DIR) / "histograms"

print("OUTPUT_DIR:", OUTPUT_DIR)
print("hist_dir:", hist_dir)
print("hist_dir absolute:", hist_dir.resolve())
print("hist_dir exists:", hist_dir.exists())

if hist_dir.exists():
    all_files = sorted(hist_dir.glob("*"))
    print("file count:", len(all_files))
    for p in all_files[:50]:
        print(p.name)

OUTPUT_DIR: ../research/110626/1226/1226_efficiency
hist_dir: ..\research\110626\1226\1226_efficiency\histograms
hist_dir absolute: C:\Users\adjop\OneDrive\Documents\golden-goose-project\research\110626\1226\1226_efficiency\histograms
hist_dir exists: True
file count: 0


In [28]:
from pathlib import Path

hist_dir = Path(OUTPUT_DIR) / "histograms"

print("OUTPUT_DIR:", OUTPUT_DIR)
print("hist_dir:", hist_dir)
print("hist_dir absolute:", hist_dir.resolve())
print("hist_dir exists:", hist_dir.exists())

if hist_dir.exists():
    all_files = sorted(hist_dir.glob("*"))
    print("file count:", len(all_files))
    for p in all_files[:50]:
        print(p.name)

OUTPUT_DIR: ../research/110626/1226/1226_efficiency
hist_dir: ..\research\110626\1226\1226_efficiency\histograms
hist_dir absolute: C:\Users\adjop\OneDrive\Documents\golden-goose-project\research\110626\1226\1226_efficiency\histograms
hist_dir exists: True
file count: 0


In [25]:
from pathlib import Path
from IPython.display import Image, display

summary_df = pd.read_csv(f"{OUTPUT_DIR}/confirmation_sweep_summary.csv")

hist_dir = Path(OUTPUT_DIR) / "histograms"

top_combos = (
    summary_df
    .sort_values(
        ["median_medium_max_R_before_stop", "profitable_2R_before_SL_rate"],
        ascending=False
    )
    .head(10)["combo_id"]
    .tolist()
)

for combo_id in top_combos:
    img_path = hist_dir / f"{combo_id}_max_R_before_stop_hist_clipped.png"
    print("Checking:", img_path, "| exists:", img_path.exists())

    if img_path.exists():
        display(Image(filename=str(img_path)))

Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0p0005__eff_0p7_max_R_before_stop_hist_clipped.png | exists: False
Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0p0005__eff_0p5_max_R_before_stop_hist_clipped.png | exists: False
Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0__eff_0p6_max_R_before_stop_hist_clipped.png | exists: False
Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0p0005__eff_0p4_max_R_before_stop_hist_clipped.png | exists: False
Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0p0005__eff_0p6_max_R_before_stop_hist_clipped.png | exists: False
Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0p0005__eff_0p9_max_R_before_stop_hist_clipped.png | exists: False
Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0__eff_0p5_max_R_before_stop_hist_clipped.png | exists: False
Checking: ..\research\110626\1226\1226_efficiency\histograms\mfe_0__eff_0p4_max_R_befo